# Phase 1 - Detection head  —  Rotax_914_ULF

Binary: is anything wrong right now? In v3 the label fires on **either** a sensor
fault **or** an engine failure mode, so this head covers both families.

Trains the shared TCN encoder from scratch; every later phase warm-starts from it.

In [7]:
import os, sys, json, time
sys.path.insert(0, os.path.abspath("."))
import numpy as np, tensorflow as tf
from tensorflow import keras
import tf_data_pipeline as P, model_architectures as A, train_common as C

C.report_gpu()

  TF 2.16.2  devices ['CPU', 'GPU']


['CPU', 'GPU']

In [8]:
# ---- SET THE ENGINE HERE. Run this notebook once per engine. ----
ENGINE = "Rotax_914_ULF"
steps = C.describe(ENGINE)
train_ds, val_ds, test_ds = C.datasets(ENGINE)

Rotax_914_ULF  (914)
  physics   : v3   TBO 2000.0 h
  rows      : 10,001,902 in 1214 scenarios
  steps/epoch: train 982  val 94  test 130
  window 128  stride 64  features 25  aux 10


### Model
TCN encoder (6 causal-dilated blocks, receptive field 253 > 128 window) + a small dense head.

In [9]:
xi  = A.build_encoder_input()
enc = A.build_encoder(xi)
out = A.build_detection_head(enc)
model = keras.Model(xi, {"y_detection": out}, name="detection")
model.summary()

Model: "detection"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ x (InputLayer)      │ (None, 128, 25)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_conv1 (Conv1D) │ (None, 128, 48)   │      3,648 │ x[0][0]           │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_drop1          │ (None, 128, 48)   │          0 │ tcn0_conv1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_conv2 (Conv1D) │ (None, 128, 48)   │      6,960 │ tcn0_drop1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_drop2          │ (None, 128, 48)   │          0 │ tcn0_conv2[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_proj (Conv1D)  │ (None, 128, 48)   │      1,248 │ x[0][0]           │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_add (Add)      │ (None, 128, 48)   │          0 │ tcn0_drop2[0][0], │
│                     │                   │            │ tcn0_proj[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_relu (ReLU)    │ (None, 128, 48)   │          0 │ tcn0_add[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_conv1 (Conv1D) │ (None, 128, 48)   │      6,960 │ tcn0_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_drop1          │ (None, 128, 48)   │          0 │ tcn1_conv1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_conv2 (Conv1D) │ (None, 128, 48)   │      6,960 │ tcn1_drop1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_drop2          │ (None, 128, 48)   │          0 │ tcn1_conv2[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_add (Add)      │ (None, 128, 48)   │          0 │ tcn1_drop2[0][0], │
│                     │                   │            │ tcn0_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_relu (ReLU)    │ (None, 128, 48)   │          0 │ tcn1_add[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_conv1 (Conv1D) │ (None, 128, 48)   │      6,960 │ tcn1_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_drop1          │ (None, 128, 48)   │          0 │ tcn2_conv1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_conv2 (Conv1D) │ (None, 128, 48)   │      6,960 │ tcn2_drop1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_drop2          │ (None, 128, 48)   │          0 │ tcn2_conv2[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_add (Add)      │ (None, 128, 48)   │          0 │ tcn2_drop2[0][0], │
│                     │                   │            │ tcn1_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_relu (ReLU)    │ (None, 128, 48)   │          0 │ tcn2_add[0][0]    │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 86,129 (336.44 KB)

 Trainable params: 86,129 (336.44 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
# (nothing extra for this phase)

In [11]:
model.compile(optimizer=keras.optimizers.Adam(1e-3, clipnorm=1.0),

              loss={"y_detection": keras.losses.BinaryCrossentropy()},

              metrics={"y_detection": [keras.metrics.BinaryAccuracy(name="acc"),

                                       keras.metrics.AUC(name="auc")]})

In [12]:
PHASE = "phase1_detection"
PREV  = None
ckpt  = C.ckpt_path(ENGINE, PHASE)
if PREV: C.warm_start(model, C.ckpt_path(ENGINE, PREV))

hist = model.fit(
    train_ds.map(C.split_labels(["y_detection"])),
    validation_data=val_ds.map(C.split_labels(["y_detection"])),
    steps_per_epoch=steps["train"],
    validation_steps=steps["val"],
    epochs=30,
    callbacks=C.callbacks(ckpt, monitor="val_loss", mode="min"),
    verbose=1,
)
print("saved:", ckpt)

Epoch 1/30


/Users/harsh/Documents/UAV_Engine/validation/venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


982/982 ━━━━━━━━━━━━━━━━━━━━ 0s 200ms/step - acc: 0.8872 - auc: 0.9345 - loss: 0.2982
Epoch 1: val_loss improved from None to 0.27222, saving model to /Users/harsh/Documents/UAV_Engine/validation/models/phase1_detection_914.keras

Epoch 1: finished saving model to /Users/harsh/Documents/UAV_Engine/validation/models/phase1_detection_914.keras
982/982 ━━━━━━━━━━━━━━━━━━━━ 211s 210ms/step - acc: 0.8872 - auc: 0.9345 - loss: 0.2982 - val_acc: 0.8944 - val_auc: 0.9561 - val_loss: 0.2722 - learning_rate: 0.0010
Epoch 2/30
982/982 ━━━━━━━━━━━━━━━━━━━━ 0s 205ms/step - acc: 0.9334 - auc: 0.9614 - loss: 0.2016
Epoch 2: val_loss improved from 0.27222 to 0.18562, saving model to /Users/harsh/Documents/UAV_Engine/validation/models/phase1_detection_914.keras

Epoch 2: finished saving model to /Users/harsh/Documents/UAV_Engine/validation/models/phase1_detection_914.keras
982/982 ━━━━━━━━━━━━━━━━━━━━ 212s 215ms/step - acc: 0.9334 - auc: 0.9614 - loss: 0.2016 - val_acc: 0.9322 - val_auc: 0.9677 - val_l

In [13]:
res = model.evaluate(test_ds.map(C.split_labels(["y_detection"])), steps=steps["test"], verbose=1, return_dict=True)



print(json.dumps({k: float(v) for k, v in res.items()}, indent=2))



with open(os.path.join(C.MODELS_DIR, f"{PHASE}_{C.ENGINE_KEY[ENGINE]}_test.json"), "w") as f:



    json.dump({k: float(v) for k, v in res.items()}, f, indent=2)

130/130 ━━━━━━━━━━━━━━━━━━━━ 14s 107ms/step - acc: 0.9303 - auc: 0.9573 - loss: 0.2078
{
  "acc": 0.9303485751152039,
  "auc": 0.957339346408844,
  "loss": 0.20783889293670654
}
